In [1]:
from pathlib import Path
import pandas as pd

print("Pandas版本：", pd.__version__)
print("当前工作目录：", Path.cwd())

Pandas版本： 3.0.3
当前工作目录： /Users/nyx/Desktop/user-behavior-analysis/notebooks


In [2]:
DATA_PATH = Path("../data/raw/user_behavior_processed.csv")

print("数据文件是否存在：", DATA_PATH.exists())
print("数据文件完整路径：", DATA_PATH.resolve())
print("文件大小：", round(DATA_PATH.stat().st_size / 1024**2, 2), "MB")

数据文件是否存在： True
数据文件完整路径： /Users/nyx/Desktop/user-behavior-analysis/data/raw/user_behavior_processed.csv
文件大小： 469.46 MB


In [3]:
sample_df = pd.read_csv(
    DATA_PATH,
    nrows=10000
)

print("样本数据形状：", sample_df.shape)
display(sample_df.head())

样本数据形状： (10000, 5)


,time,user_id,item_id,item_category,behavior_type
0,2025-12-06 02,98047837,232431562,4245,1
1,2025-12-09 20,97726136,383583590,5894,1
2,2025-12-18 11,98607707,64749712,2883,1
3,2025-12-06 10,98662432,320593836,6562,1
4,2025-12-16 21,98145908,290208520,13926,1


In [4]:
print("字段名称：")
print(sample_df.columns.tolist())

print("\n各字段的数据类型：")
print(sample_df.dtypes)

print("\n数据基本信息：")
sample_df.info()

字段名称：
['time', 'user_id', 'item_id', 'item_category', 'behavior_type']

各字段的数据类型：
time               str
user_id          int64
item_id          int64
item_category    int64
behavior_type    int64
dtype: object

数据基本信息：
<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   time           10000 non-null  str  
 1   user_id        10000 non-null  int64
 2   item_id        10000 non-null  int64
 3   item_category  10000 non-null  int64
 4   behavior_type  10000 non-null  int64
dtypes: int64(4), str(1)
memory usage: 517.7 KB


In [5]:
print("各字段缺失值数量：")
print(sample_df.isna().sum())

print("\n完全重复行数量：")
print(sample_df.duplicated().sum())

print("\n行为类型分布：")
print(sample_df["behavior_type"].value_counts().sort_index())

各字段缺失值数量：
time             0
user_id          0
item_id          0
item_category    0
behavior_type    0
dtype: int64

完全重复行数量：
717

行为类型分布：
behavior_type
1    9430
2     195
3     276
4      99
Name: count, dtype: int64


In [6]:
sample_df["time"] = pd.to_datetime(
    sample_df["time"],
    format="%Y-%m-%d %H",
    errors="coerce"
)

print("转换后的字段类型：")
print(sample_df.dtypes)

print("\n无法转换的时间数量：")
print(sample_df["time"].isna().sum())

print("\n最早时间：", sample_df["time"].min())
print("最晚时间：", sample_df["time"].max())

转换后的字段类型：
time             datetime64[us]
user_id                   int64
item_id                   int64
item_category             int64
behavior_type             int64
dtype: object

无法转换的时间数量：
0

最早时间： 2025-11-18 00:00:00
最晚时间： 2025-12-18 23:00:00


In [7]:
duplicate_rows = sample_df[
    sample_df.duplicated(keep=False)
].sort_values(
    ["user_id", "item_id", "time", "behavior_type"]
)

print("重复记录数量：", len(duplicate_rows))
display(duplicate_rows.head(20))

重复记录数量： 1395


,time,user_id,item_id,item_category,behavior_type
9845,2025-12-03 11:00:00,2967122,17123536,11178,1
9854,2025-12-03 11:00:00,2967122,17123536,11178,1
9467,2025-12-03 16:00:00,2967122,176322914,2866,1
9503,2025-12-03 16:00:00,2967122,176322914,2866,1
9431,2025-11-22 05:00:00,2967122,272639944,5027,1
9476,2025-11-22 05:00:00,2967122,272639944,5027,1
9611,2025-11-21 15:00:00,2967122,273419428,169,1
9647,2025-11-21 15:00:00,2967122,273419428,169,1
2530,2025-11-29 21:00:00,10131505,66223566,4677,1
2908,2025-11-29 21:00:00,10131505,66223566,4677,1


In [8]:
behavior_mapping = {
    1: "浏览",
    2: "收藏",
    3: "加购",
    4: "购买"
}

behavior_summary = (
    sample_df["behavior_type"]
    .value_counts()
    .sort_index()
    .rename_axis("behavior_type")
    .reset_index(name="count")
)

behavior_summary["behavior_name"] = behavior_summary["behavior_type"].map(
    behavior_mapping
)

behavior_summary["percentage"] = (
    behavior_summary["count"] / len(sample_df) * 100
).round(2)

display(behavior_summary)

,behavior_type,count,behavior_name,percentage
0,1,9430,浏览,94.30
1,2,195,收藏,1.95
2,3,276,加购,2.76
3,4,99,购买,0.99


In [9]:
dtype_mapping = {
    "user_id": "int64",
    "item_id": "int64",
    "item_category": "int32",
    "behavior_type": "int8"
}

chunk_size = 500_000

chunk_reader = pd.read_csv(
    DATA_PATH,
    dtype=dtype_mapping,
    parse_dates=["time"],
    chunksize=chunk_size
)

print("分块读取器创建成功")



分块读取器创建成功


In [10]:
total_rows = 0
missing_counts = None
behavior_counts = {}
min_time = None
max_time = None
chunk_count = 0

for chunk in pd.read_csv(
    DATA_PATH,
    dtype=dtype_mapping,
    parse_dates=["time"],
    chunksize=chunk_size
):
    chunk_count += 1
    total_rows += len(chunk)

    # 缺失值统计
    current_missing = chunk.isna().sum()

    if missing_counts is None:
        missing_counts = current_missing
    else:
        missing_counts = missing_counts.add(
            current_missing,
            fill_value=0
        )

    # 行为类型统计
    current_behavior = chunk["behavior_type"].value_counts()

    for behavior_type, count in current_behavior.items():
        behavior_counts[behavior_type] = (
            behavior_counts.get(behavior_type, 0) + count
        )

    # 时间范围
    current_min_time = chunk["time"].min()
    current_max_time = chunk["time"].max()

    if min_time is None or current_min_time < min_time:
        min_time = current_min_time

    if max_time is None or current_max_time > max_time:
        max_time = current_max_time

    print(f"已完成第 {chunk_count} 个数据块，累计 {total_rows:,} 行")

    

已完成第 1 个数据块，累计 500,000 行
已完成第 2 个数据块，累计 1,000,000 行
已完成第 3 个数据块，累计 1,500,000 行
已完成第 4 个数据块，累计 2,000,000 行
已完成第 5 个数据块，累计 2,500,000 行
已完成第 6 个数据块，累计 3,000,000 行
已完成第 7 个数据块，累计 3,500,000 行
已完成第 8 个数据块，累计 4,000,000 行
已完成第 9 个数据块，累计 4,500,000 行
已完成第 10 个数据块，累计 5,000,000 行
已完成第 11 个数据块，累计 5,500,000 行
已完成第 12 个数据块，累计 6,000,000 行
已完成第 13 个数据块，累计 6,500,000 行
已完成第 14 个数据块，累计 7,000,000 行
已完成第 15 个数据块，累计 7,500,000 行
已完成第 16 个数据块，累计 8,000,000 行
已完成第 17 个数据块，累计 8,500,000 行
已完成第 18 个数据块，累计 9,000,000 行
已完成第 19 个数据块，累计 9,500,000 行
已完成第 20 个数据块，累计 10,000,000 行
已完成第 21 个数据块，累计 10,500,000 行
已完成第 22 个数据块，累计 11,000,000 行
已完成第 23 个数据块，累计 11,500,000 行
已完成第 24 个数据块，累计 12,000,000 行
已完成第 25 个数据块，累计 12,256,906 行


In [11]:
print("完整数据总行数：", f"{total_rows:,}")
print("数据块数量：", chunk_count)

print("\n各字段缺失值：")
print(missing_counts.astype("int64"))

print("\n行为类型数量：")
for behavior_type in sorted(behavior_counts):
    print(
        behavior_type,
        behavior_mapping[behavior_type],
        f"{behavior_counts[behavior_type]:,}"
    )

print("\n最早时间：", min_time)
print("最晚时间：", max_time)

完整数据总行数： 12,256,906
数据块数量： 25

各字段缺失值：
time             0
user_id          0
item_id          0
item_category    0
behavior_type    0
dtype: int64

行为类型数量：
1 浏览 11,550,581
2 收藏 242,556
3 加购 343,564
4 购买 120,205

最早时间： 2025-11-18 00:00:00
最晚时间： 2025-12-18 23:00:00


In [12]:
full_behavior_summary = pd.DataFrame(
    {
        "behavior_type": list(behavior_counts.keys()),
        "count": list(behavior_counts.values())
    }
).sort_values("behavior_type")

full_behavior_summary["behavior_name"] = (
    full_behavior_summary["behavior_type"]
    .map(behavior_mapping)
)

full_behavior_summary["percentage"] = (
    full_behavior_summary["count"]
    / total_rows
    * 100
).round(2)

display(full_behavior_summary)


,behavior_type,count,behavior_name,percentage
0,1,11550581,浏览,94.24
2,2,242556,收藏,1.98
1,3,343564,加购,2.80
3,4,120205,购买,0.98


In [13]:
from pathlib import Path
import pandas as pd

DATA_PATH = Path("../data/raw/user_behavior_processed.csv")

dtype_mapping = {
    "user_id": "int64",
    "item_id": "int64",
    "item_category": "int32",
    "behavior_type": "int8"
}

df = pd.read_csv(
    DATA_PATH,
    dtype=dtype_mapping,
    parse_dates=["time"],
    memory_map=True
)

print("全部数据读取完成")
print("数据行列数：", df.shape)
print("总行数：", f"{len(df):,}")
print("总列数：", df.shape[1])
print(
    "内存占用：",
    round(df.memory_usage(deep=True).sum() / 1024**2, 2),
    "MB"
)



全部数据读取完成
数据行列数： (12256906, 5)
总行数： 12,256,906
总列数： 5
内存占用： 338.98 MB


In [14]:
print(df.shape)

(12256906, 5)


In [15]:
from pathlib import Path
from collections import Counter
import pandas as pd

# Notebook 位于 notebooks 文件夹时使用这个路径
DATA_PATH = Path("../data/raw/user_behavior_processed.csv")
CHUNK_SIZE = 500_000

total_rows = 0
user_ids = set()
item_ids = set()
category_ids = set()

behavior_counts = Counter()
behavior_users = {1: set(), 2: set(), 3: set(), 4: set()}

min_time = None
max_time = None
missing_counts = Counter()

columns = [
    "time",
    "user_id",
    "item_id",
    "item_category",
    "behavior_type"
]

for chunk in pd.read_csv(
    DATA_PATH,
    usecols=columns,
    chunksize=CHUNK_SIZE
):
    total_rows += len(chunk)

    # 去重主体数量
    user_ids.update(chunk["user_id"].dropna().unique())
    item_ids.update(chunk["item_id"].dropna().unique())
    category_ids.update(chunk["item_category"].dropna().unique())

    # 缺失值统计
    missing_counts.update(chunk.isna().sum().to_dict())

    # 四种行为统计
    behavior = pd.to_numeric(
        chunk["behavior_type"],
        errors="coerce"
    )

    valid_behavior = behavior.dropna().astype(int)
    behavior_counts.update(valid_behavior.tolist())

    for behavior_code in [1, 2, 3, 4]:
        users = chunk.loc[
            behavior == behavior_code, "user_id"
        ].dropna().unique()

        behavior_users[behavior_code].update(users)

    # 时间范围
    parsed_time = pd.to_datetime(
        chunk["time"],
        errors="coerce"
    ).dropna()

    if not parsed_time.empty:
        chunk_min = parsed_time.min()
        chunk_max = parsed_time.max()

        min_time = (
            chunk_min
            if min_time is None
            else min(min_time, chunk_min)
        )

        max_time = (
            chunk_max
            if max_time is None
            else max(max_time, chunk_max)
        )

# 数据概况表
overview = pd.DataFrame({
    "指标": [
        "行为记录总数",
        "去重用户数",
        "去重商品数",
        "商品品类数",
        "开始时间",
        "结束时间"
    ],
    "结果": [
        f"{total_rows:,}",
        f"{len(user_ids):,}",
        f"{len(item_ids):,}",
        f"{len(category_ids):,}",
        min_time,
        max_time
    ]
})

behavior_names = {
    1: "浏览",
    2: "收藏",
    3: "加购",
    4: "购买"
}

behavior_table = pd.DataFrame([
    {
        "行为类型": behavior_names[code],
        "行为次数": behavior_counts[code],
        "行为占比": (
            behavior_counts[code] / total_rows
            if total_rows else 0
        ),
        "涉及用户数": len(behavior_users[code])
    }
    for code in [1, 2, 3, 4]
])

behavior_table["行为次数"] = behavior_table["行为次数"].map(
    lambda x: f"{x:,}"
)
behavior_table["涉及用户数"] = behavior_table["涉及用户数"].map(
    lambda x: f"{x:,}"
)
behavior_table["行为占比"] = behavior_table["行为占比"].map(
    lambda x: f"{x:.2%}"
)

print("一、数据基础概况")
display(overview)

print("\n二、用户行为分布")
display(behavior_table)

print("\n三、各字段缺失值")
display(pd.DataFrame({
    "字段": columns,
    "缺失数量": [missing_counts[col] for col in columns]
}))


一、数据基础概况


,指标,结果
0,行为记录总数,"12,256,906"
1,去重用户数,"10,000"
2,去重商品数,"2,876,947"
3,商品品类数,"8,916"
4,开始时间,2025-11-18 00:00:00
5,结束时间,2025-12-18 23:00:00



二、用户行为分布


,行为类型,行为次数,行为占比,涉及用户数
0,浏览,"11,550,581",94.24%,"10,000"
1,收藏,"242,556",1.98%,"6,730"
2,加购,"343,564",2.80%,"8,614"
3,购买,"120,205",0.98%,"8,886"



三、各字段缺失值


,字段,缺失数量
0,time,0
1,user_id,0
2,item_id,0
3,item_category,0
4,behavior_type,0


In [16]:
from pathlib import Path
from collections import Counter
import pandas as pd

# 自动寻找原始数据
possible_paths = [
    Path("../data/raw/user_behavior_processed.csv"),
    Path("data/raw/user_behavior_processed.csv")
]

DATA_PATH = next(
    (path for path in possible_paths if path.exists()),
    None
)

if DATA_PATH is None:
    raise FileNotFoundError("没有找到 user_behavior_processed.csv")

print("读取文件：", DATA_PATH)

columns = [
    "time",
    "user_id",
    "item_id",
    "item_category",
    "behavior_type"
]

total_rows = 0
users = set()
items = set()
categories = set()
behavior_counts = Counter()

start_time = None
end_time = None

# 分块读取，不会一次性占满内存
for chunk in pd.read_csv(
    DATA_PATH,
    usecols=columns,
    chunksize=500_000
):
    total_rows += len(chunk)

    # 这里只是统计唯一数量，不是删除重复行为
    users.update(chunk["user_id"].dropna().unique())
    items.update(chunk["item_id"].dropna().unique())
    categories.update(chunk["item_category"].dropna().unique())

    behavior = pd.to_numeric(
        chunk["behavior_type"],
        errors="coerce"
    ).dropna().astype(int)

    behavior_counts.update(behavior.tolist())

    times = pd.to_datetime(
        chunk["time"],
        errors="coerce"
    ).dropna()

    if not times.empty:
        chunk_start = times.min()
        chunk_end = times.max()

        start_time = (
            chunk_start
            if start_time is None
            else min(start_time, chunk_start)
        )

        end_time = (
            chunk_end
            if end_time is None
            else max(end_time, chunk_end)
        )

print("\n===== 原始数据基础统计 =====")
print(f"行为记录总数：{total_rows:,}")
print(f"用户数量：{len(users):,}")
print(f"商品数量：{len(items):,}")
print(f"商品品类数量：{len(categories):,}")
print(f"时间范围：{start_time} 至 {end_time}")

print("\n===== 行为数量 =====")
print(f"浏览：{behavior_counts[1]:,}")
print(f"收藏：{behavior_counts[2]:,}")
print(f"加购：{behavior_counts[3]:,}")
print(f"购买：{behavior_counts[4]:,}")

读取文件： ../data/raw/user_behavior_processed.csv

===== 原始数据基础统计 =====
行为记录总数：12,256,906
用户数量：10,000
商品数量：2,876,947
商品品类数量：8,916
时间范围：2025-11-18 00:00:00 至 2025-12-18 23:00:00

===== 行为数量 =====
浏览：11,550,581
收藏：242,556
加购：343,564
购买：120,205


In [17]:
import pandas as pd

columns = [
    "time",
    "user_id",
    "item_id",
    "item_category",
    "behavior_type"
]

missing_counts = pd.Series(
    0,
    index=columns,
    dtype="int64"
)

invalid_id_counts = {
    "user_id": 0,
    "item_id": 0,
    "item_category": 0
}

invalid_behavior_count = 0
invalid_time_count = 0
problem_row_count = 0
checked_rows = 0

for chunk_number, chunk in enumerate(
    pd.read_csv(
        DATA_PATH,
        usecols=columns,
        chunksize=500_000
    ),
    start=1
):
    checked_rows += len(chunk)

    # 1. 检查缺失值
    missing_counts += chunk.isna().sum()

    # 标记本批数据中存在问题的行
    problem_mask = chunk.isna().any(axis=1)

    # 2. 检查用户、商品和品类ID
    for column in [
        "user_id",
        "item_id",
        "item_category"
    ]:
        numeric_values = pd.to_numeric(
            chunk[column],
            errors="coerce"
        )

        invalid_id_mask = (
            chunk[column].notna()
            & (
                numeric_values.isna()
                | (numeric_values <= 0)
            )
        )

        invalid_id_counts[column] += int(
            invalid_id_mask.sum()
        )

        problem_mask |= invalid_id_mask

    # 3. 检查行为编码是否属于1、2、3、4
    behavior_values = pd.to_numeric(
        chunk["behavior_type"],
        errors="coerce"
    )

    invalid_behavior_mask = (
        chunk["behavior_type"].notna()
        & ~behavior_values.isin([1, 2, 3, 4])
    )

    invalid_behavior_count += int(
        invalid_behavior_mask.sum()
    )

    problem_mask |= invalid_behavior_mask

    # 4. 检查时间能否正常解析
    parsed_time = pd.to_datetime(
        chunk["time"],
        errors="coerce"
    )

    invalid_time_mask = (
        chunk["time"].notna()
        & parsed_time.isna()
    )

    invalid_time_count += int(
        invalid_time_mask.sum()
    )

    problem_mask |= invalid_time_mask

    # 存在至少一个问题的行数
    problem_row_count += int(problem_mask.sum())

    print(
        f"已检查第 {chunk_number} 个数据块，"
        f"累计 {checked_rows:,} 行"
    )

print("\n===== 数据质量检查结果 =====")

quality_table = pd.DataFrame({
    "检查项目": [
        "总记录数",
        "存在至少一个问题的记录数",
        "user_id非法值",
        "item_id非法值",
        "item_category非法值",
        "behavior_type非法值",
        "time无法解析"
    ],
    "数量": [
        checked_rows,
        problem_row_count,
        invalid_id_counts["user_id"],
        invalid_id_counts["item_id"],
        invalid_id_counts["item_category"],
        invalid_behavior_count,
        invalid_time_count
    ]
})

display(quality_table)

print("\n===== 各字段缺失值 =====")

missing_table = pd.DataFrame({
    "字段": columns,
    "缺失数量": [
        int(missing_counts[column])
        for column in columns
    ],
    "缺失率": [
        f"{missing_counts[column] / checked_rows:.4%}"
        for column in columns
    ]
})

display(missing_table)

已检查第 1 个数据块，累计 500,000 行
已检查第 2 个数据块，累计 1,000,000 行
已检查第 3 个数据块，累计 1,500,000 行
已检查第 4 个数据块，累计 2,000,000 行
已检查第 5 个数据块，累计 2,500,000 行
已检查第 6 个数据块，累计 3,000,000 行
已检查第 7 个数据块，累计 3,500,000 行
已检查第 8 个数据块，累计 4,000,000 行
已检查第 9 个数据块，累计 4,500,000 行
已检查第 10 个数据块，累计 5,000,000 行
已检查第 11 个数据块，累计 5,500,000 行
已检查第 12 个数据块，累计 6,000,000 行
已检查第 13 个数据块，累计 6,500,000 行
已检查第 14 个数据块，累计 7,000,000 行
已检查第 15 个数据块，累计 7,500,000 行
已检查第 16 个数据块，累计 8,000,000 行
已检查第 17 个数据块，累计 8,500,000 行
已检查第 18 个数据块，累计 9,000,000 行
已检查第 19 个数据块，累计 9,500,000 行
已检查第 20 个数据块，累计 10,000,000 行
已检查第 21 个数据块，累计 10,500,000 行
已检查第 22 个数据块，累计 11,000,000 行
已检查第 23 个数据块，累计 11,500,000 行
已检查第 24 个数据块，累计 12,000,000 行
已检查第 25 个数据块，累计 12,256,906 行

===== 数据质量检查结果 =====


,检查项目,数量
0,总记录数,12256906
1,存在至少一个问题的记录数,0
2,user_id非法值,0
3,item_id非法值,0
4,item_category非法值,0
5,behavior_type非法值,0
6,time无法解析,0



===== 各字段缺失值 =====


,字段,缺失数量,缺失率
0,time,0,0.0000%
1,user_id,0,0.0000%
2,item_id,0,0.0000%
3,item_category,0,0.0000%
4,behavior_type,0,0.0000%


In [18]:
import pandas as pd

numeric_columns = [
    "user_id",
    "item_id",
    "item_category",
    "behavior_type"
]

value_ranges = {
    column: {
        "最小值": None,
        "最大值": None
    }
    for column in numeric_columns
}

for chunk_number, chunk in enumerate(
    pd.read_csv(
        DATA_PATH,
        usecols=numeric_columns,
        chunksize=500_000
    ),
    start=1
):
    for column in numeric_columns:
        values = pd.to_numeric(
            chunk[column],
            errors="raise"
        )

        current_min = int(values.min())
        current_max = int(values.max())

        old_min = value_ranges[column]["最小值"]
        old_max = value_ranges[column]["最大值"]

        value_ranges[column]["最小值"] = (
            current_min
            if old_min is None
            else min(old_min, current_min)
        )

        value_ranges[column]["最大值"] = (
            current_max
            if old_max is None
            else max(old_max, current_max)
        )

    print(f"已检查第 {chunk_number} 个数据块")

def recommend_dtype(min_value, max_value):
    if min_value >= 0:
        if max_value <= 255:
            return "uint8"
        elif max_value <= 65_535:
            return "uint16"
        elif max_value <= 4_294_967_295:
            return "uint32"
        else:
            return "uint64"
    else:
        if -128 <= min_value and max_value <= 127:
            return "int8"
        elif -32_768 <= min_value and max_value <= 32_767:
            return "int16"
        elif (
            -2_147_483_648 <= min_value
            and max_value <= 2_147_483_647
        ):
            return "int32"
        else:
            return "int64"

range_table = pd.DataFrame([
    {
        "字段": column,
        "最小值": value_ranges[column]["最小值"],
        "最大值": value_ranges[column]["最大值"],
        "建议类型": recommend_dtype(
            value_ranges[column]["最小值"],
            value_ranges[column]["最大值"]
        )
    }
    for column in numeric_columns
])

print("\n===== 字段范围与建议类型 =====")
display(range_table)


已检查第 1 个数据块
已检查第 2 个数据块
已检查第 3 个数据块
已检查第 4 个数据块
已检查第 5 个数据块
已检查第 6 个数据块
已检查第 7 个数据块
已检查第 8 个数据块
已检查第 9 个数据块
已检查第 10 个数据块
已检查第 11 个数据块
已检查第 12 个数据块
已检查第 13 个数据块
已检查第 14 个数据块
已检查第 15 个数据块
已检查第 16 个数据块
已检查第 17 个数据块
已检查第 18 个数据块
已检查第 19 个数据块
已检查第 20 个数据块
已检查第 21 个数据块
已检查第 22 个数据块
已检查第 23 个数据块
已检查第 24 个数据块
已检查第 25 个数据块

===== 字段范围与建议类型 =====


,字段,最小值,最大值,建议类型
0,user_id,4913,142455899,uint32
1,item_id,64,404562461,uint32
2,item_category,2,14080,uint16
3,behavior_type,1,4,uint8


In [19]:
from pathlib import Path
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

columns = [
    "time",
    "user_id",
    "item_id",
    "item_category",
    "behavior_type"
]

processed_dir = Path("../data/processed")
processed_dir.mkdir(parents=True, exist_ok=True)

# 避免覆盖之前生成的文件
version = 1
OUTPUT_PATH = (
    processed_dir
    / f"user_behavior_standardized_v{version}.parquet"
)

while OUTPUT_PATH.exists():
    version += 1
    OUTPUT_PATH = (
        processed_dir
        / f"user_behavior_standardized_v{version}.parquet"
    )

writer = None
written_rows = 0

try:
    for chunk_number, chunk in enumerate(
        pd.read_csv(
            DATA_PATH,
            usecols=columns,
            chunksize=500_000
        ),
        start=1
    ):
        # 统一时间格式
        chunk["time"] = pd.to_datetime(
            chunk["time"],
            errors="raise"
        )

        # 压缩字段类型
        chunk["user_id"] = chunk["user_id"].astype(
            "uint32"
        )
        chunk["item_id"] = chunk["item_id"].astype(
            "uint32"
        )
        chunk["item_category"] = chunk[
            "item_category"
        ].astype("uint16")
        chunk["behavior_type"] = chunk[
            "behavior_type"
        ].astype("uint8")

        # 固定字段顺序
        chunk = chunk[columns]

        table = pa.Table.from_pandas(
            chunk,
            preserve_index=False
        )

        if writer is None:
            writer = pq.ParquetWriter(
                OUTPUT_PATH,
                table.schema,
                compression="snappy"
            )

        writer.write_table(table)
        written_rows += len(chunk)

        print(
            f"已写入第 {chunk_number} 个数据块，"
            f"累计 {written_rows:,} 行"
        )

finally:
    if writer is not None:
        writer.close()

# 验证生成结果
parquet_file = pq.ParquetFile(OUTPUT_PATH)
verified_rows = parquet_file.metadata.num_rows

csv_size_mb = DATA_PATH.stat().st_size / 1024**2
parquet_size_mb = OUTPUT_PATH.stat().st_size / 1024**2

print("\n===== 标准化数据生成结果 =====")
print("输出文件：", OUTPUT_PATH)
print(f"写入记录数：{written_rows:,}")
print(f"Parquet验证行数：{verified_rows:,}")
print(f"原始CSV大小：{csv_size_mb:.2f} MB")
print(f"Parquet大小：{parquet_size_mb:.2f} MB")
print(f"存储空间减少：{1 - parquet_size_mb / csv_size_mb:.2%}")

print("\n===== Parquet字段结构 =====")
print(parquet_file.schema_arrow)

已写入第 1 个数据块，累计 500,000 行
已写入第 2 个数据块，累计 1,000,000 行
已写入第 3 个数据块，累计 1,500,000 行
已写入第 4 个数据块，累计 2,000,000 行
已写入第 5 个数据块，累计 2,500,000 行
已写入第 6 个数据块，累计 3,000,000 行
已写入第 7 个数据块，累计 3,500,000 行
已写入第 8 个数据块，累计 4,000,000 行
已写入第 9 个数据块，累计 4,500,000 行
已写入第 10 个数据块，累计 5,000,000 行
已写入第 11 个数据块，累计 5,500,000 行
已写入第 12 个数据块，累计 6,000,000 行
已写入第 13 个数据块，累计 6,500,000 行
已写入第 14 个数据块，累计 7,000,000 行
已写入第 15 个数据块，累计 7,500,000 行
已写入第 16 个数据块，累计 8,000,000 行
已写入第 17 个数据块，累计 8,500,000 行
已写入第 18 个数据块，累计 9,000,000 行
已写入第 19 个数据块，累计 9,500,000 行
已写入第 20 个数据块，累计 10,000,000 行
已写入第 21 个数据块，累计 10,500,000 行
已写入第 22 个数据块，累计 11,000,000 行
已写入第 23 个数据块，累计 11,500,000 行
已写入第 24 个数据块，累计 12,000,000 行
已写入第 25 个数据块，累计 12,256,906 行

===== 标准化数据生成结果 =====
输出文件： ../data/processed/user_behavior_standardized_v1.parquet
写入记录数：12,256,906
Parquet验证行数：12,256,906
原始CSV大小：469.46 MB
Parquet大小：94.38 MB
存储空间减少：79.90%

===== Parquet字段结构 =====
time: timestamp[us]
user_id: uint32
item_id: uint32
item_category: uint16
behavior_type: uint8
-- schema

In [20]:
from pathlib import Path

reports_dir = Path("../reports")
reports_dir.mkdir(parents=True, exist_ok=True)

REPORT_PATH = reports_dir / "data_quality_report.md"

behavior_names = {
    1: "浏览",
    2: "收藏",
    3: "加购",
    4: "购买"
}

behavior_lines = []

for code in [1, 2, 3, 4]:
    count = behavior_counts[code]
    ratio = count / total_rows

    behavior_lines.append(
        f"| {code} | {behavior_names[code]} | "
        f"{count:,} | {ratio:.2%} |"
    )

behavior_table_text = "\n".join(behavior_lines)

report_text = f"""# 淘宝用户行为数据质量校验报告

## 1. 数据基础概况

| 指标 | 统计结果 |
|---|---:|
| 行为记录总数 | {total_rows:,} |
| 用户数量 | {len(users):,} |
| 商品数量 | {len(items):,} |
| 商品品类数量 | {len(categories):,} |
| 开始时间 | {start_time} |
| 结束时间 | {end_time} |

## 2. 用户行为分布

| 行为编码 | 行为类型 | 行为次数 | 行为占比 |
|---:|---|---:|---:|
{behavior_table_text}

说明：行为占比反映各类行为记录在全部事件中的比例，
不直接等同于用户转化率。

## 3. 数据完整性检查

| 检查项目 | 结果 |
|---|---:|
| 存在至少一个问题的记录数 | {problem_row_count:,} |
| time缺失值 | {int(missing_counts["time"]):,} |
| user_id缺失值 | {int(missing_counts["user_id"]):,} |
| item_id缺失值 | {int(missing_counts["item_id"]):,} |
| item_category缺失值 | {int(missing_counts["item_category"]):,} |
| behavior_type缺失值 | {int(missing_counts["behavior_type"]):,} |
| user_id非法值 | {invalid_id_counts["user_id"]:,} |
| item_id非法值 | {invalid_id_counts["item_id"]:,} |
| item_category非法值 | {invalid_id_counts["item_category"]:,} |
| behavior_type非法值 | {invalid_behavior_count:,} |
| 无法解析的时间记录 | {invalid_time_count:,} |

## 4. 字段标准化结果

| 字段 | 标准化类型 |
|---|---|
| time | timestamp |
| user_id | uint32 |
| item_id | uint32 |
| item_category | uint16 |
| behavior_type | uint8 |

## 5. 重复行为处理原则

本阶段未直接删除任何重复行为记录。

同一用户对同一商品产生的重复浏览、收藏、加购和购买行为
具有实际业务意义，可以反映用户兴趣强度、购买意愿、
决策犹豫程度和复购行为。

后续特征工程将基于这些记录构建：

- 重复浏览次数；
- 重复收藏次数；
- 重复加购次数；
- 重复购买次数；
- 行为频率；
- 行为时间间隔。

## 6. 存储优化结果

| 指标 | 结果 |
|---|---:|
| 原始CSV大小 | {csv_size_mb:.2f} MB |
| Parquet文件大小 | {parquet_size_mb:.2f} MB |
| 存储空间减少 | {1 - parquet_size_mb / csv_size_mb:.2%} |
| Parquet记录数 | {verified_rows:,} |

## 7. 校验结论

原始数据共包含{total_rows:,}条用户行为记录。
所有核心字段均不存在缺失值或非法值，时间字段能够正常解析，
四类行为记录之和与数据总行数一致。

标准化处理完整保留了原始行为记录，并通过字段类型压缩和
Parquet列式存储将文件体积减少
{1 - parquet_size_mb / csv_size_mb:.2%}。

注意：数据文件的时间范围显示为2025年11月18日至12月18日，
最终使用前需要根据数据来源说明确认日期是否经过匿名化或平移。
"""

REPORT_PATH.write_text(
    report_text,
    encoding="utf-8"
)

print("数据质量报告已生成：")
print(REPORT_PATH)
print("\n===== 报告预览 =====\n")
print(report_text)

数据质量报告已生成：
../reports/data_quality_report.md

===== 报告预览 =====

# 淘宝用户行为数据质量校验报告

## 1. 数据基础概况

| 指标 | 统计结果 |
|---|---:|
| 行为记录总数 | 12,256,906 |
| 用户数量 | 10,000 |
| 商品数量 | 2,876,947 |
| 商品品类数量 | 8,916 |
| 开始时间 | 2025-11-18 00:00:00 |
| 结束时间 | 2025-12-18 23:00:00 |

## 2. 用户行为分布

| 行为编码 | 行为类型 | 行为次数 | 行为占比 |
|---:|---|---:|---:|
| 1 | 浏览 | 11,550,581 | 94.24% |
| 2 | 收藏 | 242,556 | 1.98% |
| 3 | 加购 | 343,564 | 2.80% |
| 4 | 购买 | 120,205 | 0.98% |

说明：行为占比反映各类行为记录在全部事件中的比例，
不直接等同于用户转化率。

## 3. 数据完整性检查

| 检查项目 | 结果 |
|---|---:|
| 存在至少一个问题的记录数 | 0 |
| time缺失值 | 0 |
| user_id缺失值 | 0 |
| item_id缺失值 | 0 |
| item_category缺失值 | 0 |
| behavior_type缺失值 | 0 |
| user_id非法值 | 0 |
| item_id非法值 | 0 |
| item_category非法值 | 0 |
| behavior_type非法值 | 0 |
| 无法解析的时间记录 | 0 |

## 4. 字段标准化结果

| 字段 | 标准化类型 |
|---|---|
| time | timestamp |
| user_id | uint32 |
| item_id | uint32 |
| item_category | uint16 |
| behavior_type | uint8 |

## 5. 重复行为处理原则

本阶段未直接删除任何重复行为记录。

同一用户对同一商品产生的重复浏览、收藏、加购和购买行为
具有实际业务意义，可以反映用户兴

In [21]:
from pathlib import Path
import subprocess

PROJECT_ROOT = Path("..").resolve()
GITIGNORE_PATH = PROJECT_ROOT / ".gitignore"

print("===== 当前 .gitignore =====")

if GITIGNORE_PATH.exists():
    print(GITIGNORE_PATH.read_text(encoding="utf-8"))
else:
    print("没有找到 .gitignore")

print("\n===== 当前 Git 状态 =====")

result = subprocess.run(
    ["git", "status", "--short"],
    cwd=PROJECT_ROOT,
    capture_output=True,
    text=True
)

print(result.stdout or "当前没有未提交的改动")

===== 当前 .gitignore =====
.venv/
__pycache__/
*.pyc
.DS_Store

.ipynb_checkpoints/
**/.ipynb_checkpoints/

data/raw/*
!data/raw/.gitkeep

data/processed/*
!data/processed/.gitkeep

outputs/*
!outputs/.gitkeep

*.db


===== 当前 Git 状态 =====
 M notebooks/01_data_loading.ipynb
?? reports/data_quality_report.md



In [22]:
import pyarrow.parquet as pq

parquet_path = "../data/processed/user_behavior_standardized_v1.parquet"

preview = (
    pq.ParquetFile(parquet_path)
    .read_row_group(0)
    .slice(0, 10)
    .to_pandas()
)

display(preview)



,time,user_id,item_id,item_category,behavior_type
0,2025-12-06 02:00:00,98047837,232431562,4245,1
1,2025-12-09 20:00:00,97726136,383583590,5894,1
2,2025-12-18 11:00:00,98607707,64749712,2883,1
3,2025-12-06 10:00:00,98662432,320593836,6562,1
4,2025-12-16 21:00:00,98145908,290208520,13926,1
5,2025-12-03 20:00:00,93784494,337869048,3979,1
6,2025-12-13 20:00:00,94832743,105749725,9559,1
7,2025-11-27 16:00:00,95290487,76866650,10875,1
8,2025-12-11 23:00:00,96610296,161166643,3064,1
9,2025-12-05 23:00:00,100684618,21751142,2158,3


In [24]:
%pip install duckdb

Note: you may need to restart the kernel to use updated packages.


In [25]:
from pathlib import Path
import duckdb

PARQUET_PATH = Path(
    "../data/processed/"
    "user_behavior_standardized_v1.parquet"
).resolve()

DATABASE_PATH = Path(
    "../data/processed/"
    "user_behavior_analytics.db"
).resolve()

# 建立DuckDB数据库连接
con = duckdb.connect(str(DATABASE_PATH))

parquet_sql_path = PARQUET_PATH.as_posix()

# 创建用户维度中间表
con.execute(f"""
CREATE OR REPLACE TABLE user_summary AS
SELECT
    user_id,

    COUNT(*) AS total_actions,

    COUNT(*) FILTER (
        WHERE behavior_type = 1
    ) AS view_count,

    COUNT(*) FILTER (
        WHERE behavior_type = 2
    ) AS favorite_count,

    COUNT(*) FILTER (
        WHERE behavior_type = 3
    ) AS cart_count,

    COUNT(*) FILTER (
        WHERE behavior_type = 4
    ) AS purchase_count,

    COUNT(DISTINCT item_id) AS unique_items,

    COUNT(DISTINCT item_category)
        AS unique_categories,

    COUNT(DISTINCT CAST(time AS DATE))
        AS active_days,

    MIN(time) AS first_action_time,

    MAX(time) AS last_action_time,

    COUNT(DISTINCT CASE
        WHEN behavior_type = 4
        THEN item_id
    END) AS purchased_items,

    ROUND(
        100.0
        * COUNT(*) FILTER (
            WHERE behavior_type = 4
        )
        / COUNT(*),
        4
    ) AS purchase_action_pct

FROM read_parquet('{parquet_sql_path}')

GROUP BY user_id
""")

# 为用户ID建立唯一索引
con.execute("""
CREATE UNIQUE INDEX IF NOT EXISTS
idx_user_summary_user_id
ON user_summary(user_id)
""")

# 验证中间表
validation = con.execute("""
SELECT
    COUNT(*) AS user_rows,
    SUM(total_actions) AS restored_actions,
    MIN(first_action_time) AS earliest_time,
    MAX(last_action_time) AS latest_time
FROM user_summary
""").fetchdf()

preview = con.execute("""
SELECT *
FROM user_summary
ORDER BY total_actions DESC
LIMIT 10
""").fetchdf()

print("===== 用户中间表校验 =====")
display(validation)

print("\n===== 最活跃的10名用户 =====")
display(preview)


===== 用户中间表校验 =====


,user_rows,restored_actions,earliest_time,latest_time
0,10000,12256906.0,2025-11-18,2025-12-18 23:00:00



===== 最活跃的10名用户 =====


,user_id,total_actions,view_count,favorite_count,cart_count,purchase_count,unique_items,unique_categories,active_days,first_action_time,last_action_time,purchased_items,purchase_action_pct
0,36233277,31030,27720,2935,328,47,11567,372,31,2025-11-18 07:00:00,2025-12-18 23:00:00,43,0.1515
1,65645933,20770,18124,2600,27,19,7223,325,30,2025-11-18 03:00:00,2025-12-18 22:00:00,13,0.0915
2,73196588,20146,20146,0,0,0,14011,1131,31,2025-11-18 00:00:00,2025-12-18 23:00:00,0,0.0000
3,59511789,20129,18558,1274,263,34,6730,320,31,2025-11-18 10:00:00,2025-12-18 23:00:00,30,0.1689
4,130270245,19786,18857,726,185,18,5672,419,28,2025-11-18 00:00:00,2025-12-18 23:00:00,18,0.0910
5,7234861,17574,15791,1165,569,49,6149,619,31,2025-11-18 00:00:00,2025-12-18 19:00:00,48,0.2788
6,83813302,17248,16874,277,81,16,6752,311,31,2025-11-18 09:00:00,2025-12-18 22:00:00,15,0.0928
7,52577851,13419,12555,10,792,62,4757,267,31,2025-11-18 08:00:00,2025-12-18 21:00:00,51,0.4620
8,137175187,12908,12339,254,284,31,4291,217,31,2025-11-18 01:00:00,2025-12-18 23:00:00,25,0.2402
9,123842164,12705,11342,168,546,649,4053,239,31,2025-11-18 00:00:00,2025-12-18 23:00:00,299,5.1082


In [26]:
# 创建商品维度中间表
con.execute(f"""
CREATE OR REPLACE TABLE item_summary AS
SELECT
    item_id,

    MIN(item_category) AS item_category,

    COUNT(DISTINCT item_category)
        AS category_count,

    COUNT(*) AS total_actions,

    COUNT(*) FILTER (
        WHERE behavior_type = 1
    ) AS view_count,

    COUNT(*) FILTER (
        WHERE behavior_type = 2
    ) AS favorite_count,

    COUNT(*) FILTER (
        WHERE behavior_type = 3
    ) AS cart_count,

    COUNT(*) FILTER (
        WHERE behavior_type = 4
    ) AS purchase_count,

    COUNT(DISTINCT user_id)
        AS unique_users,

    COUNT(DISTINCT CASE
        WHEN behavior_type = 1
        THEN user_id
    END) AS unique_viewers,

    COUNT(DISTINCT CASE
        WHEN behavior_type = 2
        THEN user_id
    END) AS unique_favoriters,

    COUNT(DISTINCT CASE
        WHEN behavior_type = 3
        THEN user_id
    END) AS unique_cart_users,

    COUNT(DISTINCT CASE
        WHEN behavior_type = 4
        THEN user_id
    END) AS unique_buyers,

    MIN(time) AS first_action_time,

    MAX(time) AS last_action_time,

    ROUND(
        100.0
        * COUNT(*) FILTER (
            WHERE behavior_type = 4
        )
        / COUNT(*),
        4
    ) AS purchase_action_pct

FROM read_parquet('{parquet_sql_path}')

GROUP BY item_id
""")

# 为商品ID建立唯一索引
con.execute("""
CREATE UNIQUE INDEX IF NOT EXISTS
idx_item_summary_item_id
ON item_summary(item_id)
""")

# 校验商品表
item_validation = con.execute("""
SELECT
    COUNT(*) AS item_rows,
    SUM(total_actions) AS restored_actions,
    COUNT(*) FILTER (
        WHERE category_count > 1
    ) AS items_with_multiple_categories
FROM item_summary
""").fetchdf()

# 查看行为量最高的10件商品
top_items = con.execute("""
SELECT *
FROM item_summary
ORDER BY total_actions DESC
LIMIT 10
""").fetchdf()

print("===== 商品中间表校验 =====")
display(item_validation)

print("\n===== 行为量最高的10件商品 =====")
display(top_items)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

===== 商品中间表校验 =====


,item_rows,restored_actions,items_with_multiple_categories
0,2876947,12256906.0,0



===== 行为量最高的10件商品 =====


,item_id,item_category,category_count,total_actions,view_count,favorite_count,cart_count,purchase_count,unique_users,unique_viewers,unique_favoriters,unique_cart_users,unique_buyers,first_action_time,last_action_time,purchase_action_pct
0,112921337,6000,1,1445,1431,5,8,1,518,518,5,8,1,2025-11-18 00:00:00,2025-12-18 22:00:00,0.0692
1,97655171,6000,1,1289,1249,7,20,13,332,332,7,16,12,2025-11-18 00:00:00,2025-12-18 22:00:00,1.0085
2,387911330,6000,1,1102,1053,10,20,19,298,298,8,13,15,2025-11-18 05:00:00,2025-12-18 22:00:00,1.7241
3,135104537,6000,1,931,916,7,8,0,339,339,6,8,0,2025-11-18 00:00:00,2025-12-18 22:00:00,0.0000
4,14087919,5232,1,823,740,15,33,35,206,205,12,29,25,2025-11-18 09:00:00,2025-12-18 23:00:00,4.2527
5,277922302,5399,1,808,763,24,20,1,275,275,23,15,1,2025-11-18 13:00:00,2025-12-18 23:00:00,0.1238
6,2217535,6000,1,805,792,1,11,1,317,317,1,7,1,2025-11-18 00:00:00,2025-12-18 21:00:00,0.1242
7,128186279,3381,1,795,765,13,16,1,341,341,13,11,1,2025-11-18 00:00:00,2025-12-18 22:00:00,0.1258
8,5685392,3381,1,793,767,12,12,2,302,302,11,10,2,2025-11-18 01:00:00,2025-12-12 21:00:00,0.2522
9,209323160,10072,1,744,716,2,26,0,282,282,2,25,0,2025-12-09 00:00:00,2025-12-17 22:00:00,0.0000


In [27]:
# 创建小时级时间维度中间表
con.execute(f"""
CREATE OR REPLACE TABLE time_summary AS
SELECT
    time,

    CAST(time AS DATE) AS action_date,

    CAST(
        DATE_TRUNC('week', time)
        AS DATE
    ) AS week_start,

    EXTRACT(HOUR FROM time)
        AS hour_of_day,

    EXTRACT(DOW FROM time)
        AS day_of_week,

    CASE
        WHEN EXTRACT(DOW FROM time) IN (0, 6)
        THEN TRUE
        ELSE FALSE
    END AS is_weekend,

    COUNT(*) AS total_actions,

    COUNT(*) FILTER (
        WHERE behavior_type = 1
    ) AS view_count,

    COUNT(*) FILTER (
        WHERE behavior_type = 2
    ) AS favorite_count,

    COUNT(*) FILTER (
        WHERE behavior_type = 3
    ) AS cart_count,

    COUNT(*) FILTER (
        WHERE behavior_type = 4
    ) AS purchase_count,

    COUNT(DISTINCT user_id)
        AS active_users,

    COUNT(DISTINCT item_id)
        AS active_items,

    ROUND(
        100.0
        * COUNT(*) FILTER (
            WHERE behavior_type = 4
        )
        / COUNT(*),
        4
    ) AS purchase_action_pct

FROM read_parquet('{parquet_sql_path}')

GROUP BY time
""")

# 为时间建立唯一索引
con.execute("""
CREATE UNIQUE INDEX IF NOT EXISTS
idx_time_summary_time
ON time_summary(time)
""")

# 校验时间表
time_validation = con.execute("""
SELECT
    COUNT(*) AS hourly_rows,
    SUM(total_actions) AS restored_actions,
    MIN(time) AS earliest_time,
    MAX(time) AS latest_time
FROM time_summary
""").fetchdf()

# 查看行为量最高的10个小时
top_hours = con.execute("""
SELECT *
FROM time_summary
ORDER BY total_actions DESC
LIMIT 10
""").fetchdf()

print("===== 时间中间表校验 =====")
display(time_validation)

print("\n===== 行为量最高的10个小时 =====")
display(top_hours)

===== 时间中间表校验 =====


,hourly_rows,restored_actions,earliest_time,latest_time
0,744,12256906.0,2025-11-18,2025-12-18 23:00:00



===== 行为量最高的10个小时 =====


,time,action_date,week_start,hour_of_day,day_of_week,is_weekend,total_actions,view_count,favorite_count,cart_count,purchase_count,active_users,active_items,purchase_action_pct
0,2025-12-11 22:00:00,2025-12-11,2025-12-08,22,4,False,54797,51508,946,2011,332,2156,24235,0.6059
1,2025-12-12 22:00:00,2025-12-12,2025-12-08,22,5,False,51337,47748,709,1895,985,2178,22055,1.9187
2,2025-12-12 21:00:00,2025-12-12,2025-12-08,21,5,False,51259,47831,735,1843,850,2287,21861,1.6582
3,2025-12-12 00:00:00,2025-12-12,2025-12-08,0,5,False,50030,45229,584,2267,1950,1569,20261,3.8977
4,2025-12-11 23:00:00,2025-12-11,2025-12-08,23,4,False,49276,46151,958,1969,198,1706,21603,0.4018
5,2025-12-11 21:00:00,2025-12-11,2025-12-08,21,4,False,47802,45095,784,1640,283,2136,21024,0.5920
6,2025-12-12 23:00:00,2025-12-12,2025-12-08,23,5,False,46576,43091,669,1866,950,1728,20145,2.0397
7,2025-12-12 20:00:00,2025-12-12,2025-12-08,20,5,False,43206,40770,565,1228,643,2103,18506,1.4882
8,2025-12-10 21:00:00,2025-12-10,2025-12-08,21,3,False,39023,36888,782,1144,209,1718,17370,0.5356
9,2025-12-11 20:00:00,2025-12-11,2025-12-08,20,4,False,38918,36739,684,1232,263,1838,17251,0.6758


In [28]:
# 1. 日维度中间表
con.execute(f"""
CREATE OR REPLACE TABLE daily_summary AS
SELECT
    CAST(time AS DATE) AS action_date,

    EXTRACT(DOW FROM time)
        AS day_of_week,

    CASE
        WHEN EXTRACT(DOW FROM time) IN (0, 6)
        THEN TRUE
        ELSE FALSE
    END AS is_weekend,

    COUNT(*) AS total_actions,

    COUNT(*) FILTER (
        WHERE behavior_type = 1
    ) AS view_count,

    COUNT(*) FILTER (
        WHERE behavior_type = 2
    ) AS favorite_count,

    COUNT(*) FILTER (
        WHERE behavior_type = 3
    ) AS cart_count,

    COUNT(*) FILTER (
        WHERE behavior_type = 4
    ) AS purchase_count,

    COUNT(DISTINCT user_id)
        AS active_users,

    COUNT(DISTINCT item_id)
        AS active_items,

    ROUND(
        100.0
        * COUNT(*) FILTER (
            WHERE behavior_type = 4
        )
        / COUNT(*),
        4
    ) AS purchase_action_pct

FROM read_parquet('{parquet_sql_path}')

GROUP BY
    CAST(time AS DATE),
    EXTRACT(DOW FROM time)
""")

# 2. 周维度中间表
con.execute(f"""
CREATE OR REPLACE TABLE weekly_summary AS
SELECT
    CAST(
        DATE_TRUNC('week', time)
        AS DATE
    ) AS week_start,

    MIN(CAST(time AS DATE))
        AS observed_start_date,

    MAX(CAST(time AS DATE))
        AS observed_end_date,

    COUNT(DISTINCT CAST(time AS DATE))
        AS covered_days,

    COUNT(*) AS total_actions,

    COUNT(*) FILTER (
        WHERE behavior_type = 1
    ) AS view_count,

    COUNT(*) FILTER (
        WHERE behavior_type = 2
    ) AS favorite_count,

    COUNT(*) FILTER (
        WHERE behavior_type = 3
    ) AS cart_count,

    COUNT(*) FILTER (
        WHERE behavior_type = 4
    ) AS purchase_count,

    COUNT(DISTINCT user_id)
        AS active_users,

    COUNT(DISTINCT item_id)
        AS active_items,

    ROUND(
        100.0
        * COUNT(*) FILTER (
            WHERE behavior_type = 4
        )
        / COUNT(*),
        4
    ) AS purchase_action_pct

FROM read_parquet('{parquet_sql_path}')

GROUP BY
    DATE_TRUNC('week', time)
""")

# 建立索引
con.execute("""
CREATE UNIQUE INDEX IF NOT EXISTS
idx_daily_summary_date
ON daily_summary(action_date)
""")

con.execute("""
CREATE UNIQUE INDEX IF NOT EXISTS
idx_weekly_summary_week
ON weekly_summary(week_start)
""")

# 联合校验
time_grain_validation = con.execute("""
SELECT
    (SELECT COUNT(*) FROM time_summary)
        AS hourly_rows,

    (SELECT COUNT(*) FROM daily_summary)
        AS daily_rows,

    (SELECT COUNT(*) FROM weekly_summary)
        AS weekly_rows,

    (SELECT SUM(total_actions)
     FROM daily_summary)
        AS daily_restored_actions,

    (SELECT SUM(total_actions)
     FROM weekly_summary)
        AS weekly_restored_actions
""").fetchdf()

daily_preview = con.execute("""
SELECT *
FROM daily_summary
ORDER BY total_actions DESC
LIMIT 10
""").fetchdf()

weekly_preview = con.execute("""
SELECT *
FROM weekly_summary
ORDER BY week_start
""").fetchdf()

print("===== 时间粒度联合校验 =====")
display(time_grain_validation)

print("\n===== 行为量最高的10天 =====")
display(daily_preview)

print("\n===== 周度汇总 =====")
display(weekly_preview)

===== 时间粒度联合校验 =====


,hourly_rows,daily_rows,weekly_rows,daily_restored_actions,weekly_restored_actions
0,744,31,5,12256906.0,12256906.0



===== 行为量最高的10天 =====


,action_date,day_of_week,is_weekend,total_actions,view_count,favorite_count,cart_count,purchase_count,active_users,active_items,purchase_action_pct
0,2025-12-12,5,False,691712,641507,10446,24508,15251,7720,232398,2.2048
1,2025-12-11,4,False,488508,460329,9310,15643,3226,6894,180622,0.6604
2,2025-12-10,3,False,421910,397661,8753,12280,3216,6652,162006,0.7622
3,2025-12-03,3,False,411606,387497,8492,11732,3885,6585,156706,0.9439
4,2025-12-13,6,True,407160,385337,7859,10486,3478,6776,156418,0.8542
5,2025-12-02,2,False,405216,382052,8022,11521,3621,6550,154480,0.8936
6,2025-12-14,0,True,402541,380717,8128,10213,3483,6668,155900,0.8653
7,2025-11-30,0,True,401620,379439,7786,10779,3616,6379,155985,0.9004
8,2025-12-04,4,False,399952,376307,8559,11395,3691,6531,153369,0.9229
9,2025-12-07,0,True,399751,376596,8632,11267,3256,6422,152758,0.8145



===== 周度汇总 =====


,week_start,observed_start_date,observed_end_date,covered_days,total_actions,view_count,favorite_count,cart_count,purchase_count,active_users,active_items,purchase_action_pct
0,2025-11-17,2025-11-18,2025-11-23,6,2156114,2032873,43009,59416,20816,8964,681413,0.9654
1,2025-11-24,2025-11-24,2025-11-30,7,2587816,2442624,51332,69682,24178,9224,801899,0.9343
2,2025-12-01,2025-12-01,2025-12-07,7,2762624,2602649,56821,78201,24953,9354,832895,0.9032
3,2025-12-08,2025-12-08,2025-12-14,7,3196523,3003909,61284,95808,35522,9506,920810,1.1113
4,2025-12-15,2025-12-15,2025-12-18,4,1553829,1468526,30110,40457,14736,8947,514725,0.9484


In [29]:
from pathlib import Path

processed_dir = Path("../data/processed").resolve()
reports_dir = Path("../reports").resolve()

processed_dir.mkdir(parents=True, exist_ok=True)
reports_dir.mkdir(parents=True, exist_ok=True)

# 需要导出的中间表
export_jobs = {
    "user_summary": "user_summary_v1.parquet",
    "item_summary": "item_summary_v1.parquet",
    "time_summary": "time_hourly_summary_v1.parquet",
    "daily_summary": "time_daily_summary_v1.parquet",
    "weekly_summary": "time_weekly_summary_v1.parquet"
}

print("===== 导出中间表 =====")

for table_name, file_name in export_jobs.items():
    output_path = processed_dir / file_name

    if output_path.exists():
        print(f"已存在，跳过：{file_name}")
    else:
        sql_output_path = output_path.as_posix()

        con.execute(f"""
        COPY {table_name}
        TO '{sql_output_path}'
        (FORMAT PARQUET, COMPRESSION SNAPPY)
        """)

        print(f"导出成功：{file_name}")

# 获取各表行数
table_counts = {}

for table_name in export_jobs:
    row_count = con.execute(
        f"SELECT COUNT(*) FROM {table_name}"
    ).fetchone()[0]

    table_counts[table_name] = row_count

# 生成中间表设计说明
DESIGN_REPORT_PATH = (
    reports_dir
    / "intermediate_table_design.md"
)

design_report = f"""# 多维度中间表设计说明

## 1. 设计目标

基于标准化用户行为数据构建面向分析的聚合中间表，
降低后续探索性分析、特征工程和模型训练的重复计算成本。

原始行为明细不会删除，仍保存在标准化基础数据集中。
重复浏览、收藏、加购和购买行为在中间表中体现为行为次数。

## 2. 数据模型

标准化行为明细表是核心事实表，包含以下关联字段：

- `user_id`
- `item_id`
- `item_category`
- `time`
- `behavior_type`

不同中间表使用与自身统计粒度相匹配的主键，
不使用统一的复合主键。

## 3. 中间表说明

| 表名 | 统计粒度 | 主键 | 行数 | 主要指标 |
|---|---|---|---:|---|
| user_summary | 每名用户一行 | user_id | {table_counts["user_summary"]:,} | 浏览、收藏、加购、购买、活跃天数、商品覆盖 |
| item_summary | 每件商品一行 | item_id | {table_counts["item_summary"]:,} | 流量、互动、购买、独立用户和买家数 |
| time_summary | 每小时一行 | time | {table_counts["time_summary"]:,} | 小时行为量、活跃用户和活跃商品 |
| daily_summary | 每天一行 | action_date | {table_counts["daily_summary"]:,} | 日行为趋势和购买行为占比 |
| weekly_summary | 每周一行 | week_start | {table_counts["weekly_summary"]:,} | 周度趋势和覆盖天数 |

## 4. 关联方式

- 行为明细表通过 `user_id` 关联用户中间表；
- 行为明细表通过 `item_id` 关联商品中间表；
- 行为明细表通过 `time` 关联小时中间表；
- 通过日期转换关联日表和周表；
- 商品的品类字段为 `item_category`。

## 5. 重复行为处理

没有直接删除重复行为。

同一用户对同一商品的重复行为被汇总为：

- `view_count`
- `favorite_count`
- `cart_count`
- `purchase_count`

这些指标反映兴趣强度、购买意愿和复购特征。

## 6. 指标口径说明

`purchase_action_pct` 的计算方式为：

购买行为次数 ÷ 全部行为次数 × 100%

该指标表示购买行为在全部行为中的占比，
不直接等同于严格意义上的用户购买转化率。

## 7. 数据一致性校验

| 校验项目 | 结果 |
|---|---:|
| 用户表行数 | {table_counts["user_summary"]:,} |
| 商品表行数 | {table_counts["item_summary"]:,} |
| 小时表行数 | {table_counts["time_summary"]:,} |
| 日表行数 | {table_counts["daily_summary"]:,} |
| 周表行数 | {table_counts["weekly_summary"]:,} |
| 用户表还原行为数 | 12,256,906 |
| 商品表还原行为数 | 12,256,906 |
| 日表还原行为数 | 12,256,906 |
| 周表还原行为数 | 12,256,906 |
| 多品类归属商品数 | 0 |

## 8. 时间分析注意事项

数据覆盖2025年11月18日至12月18日。

首个自然周只覆盖6天，最后一个自然周只覆盖4天，
因此进行周度比较时应使用日均行为量，不能只比较周度总量。

12月12日的行为量和购买行为占比明显升高，
与“双12”时间窗口重合，后续需结合活动背景进一步验证，
不能仅凭时间重合直接认定因果关系。
"""

DESIGN_REPORT_PATH.write_text(
    design_report,
    encoding="utf-8"
)

print("\n===== 步骤二产出物 =====")

for file_name in export_jobs.values():
    path = processed_dir / file_name
    size_mb = path.stat().st_size / 1024**2

    print(f"{file_name}: {size_mb:.2f} MB")

print(
    "设计说明：",
    DESIGN_REPORT_PATH
)

===== 导出中间表 =====
导出成功：user_summary_v1.parquet
导出成功：item_summary_v1.parquet
导出成功：time_hourly_summary_v1.parquet
导出成功：time_daily_summary_v1.parquet
导出成功：time_weekly_summary_v1.parquet

===== 步骤二产出物 =====
user_summary_v1.parquet: 0.27 MB
item_summary_v1.parquet: 31.72 MB
time_hourly_summary_v1.parquet: 0.03 MB
time_daily_summary_v1.parquet: 0.00 MB
time_weekly_summary_v1.parquet: 0.00 MB
设计说明： /Users/nyx/Desktop/user-behavior-analysis/reports/intermediate_table_design.md


In [30]:
from pathlib import Path

# 构建品类维度中间表
con.execute(f"""
CREATE OR REPLACE TABLE category_summary AS
SELECT
    item_category,

    COUNT(*) AS total_actions,

    COUNT(*) FILTER (
        WHERE behavior_type = 1
    ) AS view_count,

    COUNT(*) FILTER (
        WHERE behavior_type = 2
    ) AS favorite_count,

    COUNT(*) FILTER (
        WHERE behavior_type = 3
    ) AS cart_count,

    COUNT(*) FILTER (
        WHERE behavior_type = 4
    ) AS purchase_count,

    COUNT(DISTINCT user_id)
        AS unique_users,

    COUNT(DISTINCT item_id)
        AS unique_items,

    COUNT(DISTINCT CASE
        WHEN behavior_type = 4
        THEN user_id
    END) AS unique_buyers,

    COUNT(DISTINCT CASE
        WHEN behavior_type = 4
        THEN item_id
    END) AS purchased_items,

    MIN(time) AS first_action_time,

    MAX(time) AS last_action_time,

    ROUND(
        100.0
        * COUNT(*) FILTER (
            WHERE behavior_type = 4
        )
        / COUNT(*),
        4
    ) AS purchase_action_pct

FROM read_parquet('{parquet_sql_path}')

GROUP BY item_category
""")

# 建立唯一索引
con.execute("""
CREATE UNIQUE INDEX IF NOT EXISTS
idx_category_summary_category
ON category_summary(item_category)
""")

# 数据一致性校验
category_validation = con.execute("""
SELECT
    COUNT(*) AS category_rows,
    SUM(total_actions) AS restored_actions,
    SUM(unique_items) AS restored_items,
    MIN(first_action_time) AS earliest_time,
    MAX(last_action_time) AS latest_time
FROM category_summary
""").fetchdf()

# 查看行为量最高的10个品类
top_categories = con.execute("""
SELECT *
FROM category_summary
ORDER BY total_actions DESC
LIMIT 10
""").fetchdf()

print("===== 品类中间表校验 =====")
display(category_validation)

print("\n===== 行为量最高的10个品类 =====")
display(top_categories)

# 导出完整品类表
category_output = Path(
    "../data/processed/category_summary_v1.parquet"
)

if category_output.exists():
    print("\n文件已经存在：", category_output)
else:
    con.execute("""
    COPY category_summary
    TO '../data/processed/category_summary_v1.parquet'
    (FORMAT PARQUET, COMPRESSION SNAPPY)
    """)

    print("\n品类表已导出：", category_output)

===== 品类中间表校验 =====


,category_rows,restored_actions,restored_items,earliest_time,latest_time
0,8916,12256906.0,2876947.0,2025-11-18,2025-12-18 23:00:00



===== 行为量最高的10个品类 =====


,item_category,total_actions,view_count,favorite_count,cart_count,purchase_count,unique_users,unique_items,unique_buyers,purchased_items,first_action_time,last_action_time,purchase_action_pct
0,1863,393247,371738,10200,9309,2000,5997,83354,1252,1674,2025-11-18,2025-12-18 23:00:00,0.5086
1,13230,356773,342694,7226,6012,841,5619,74067,564,763,2025-11-18,2025-12-18 23:00:00,0.2357
2,5027,334280,320870,6988,5564,858,5499,60323,627,739,2025-11-18,2025-12-18 23:00:00,0.2567
3,5894,330207,314784,7850,6615,958,5279,83758,605,884,2025-11-18,2025-12-18 23:00:00,0.2901
4,6513,295768,281370,6688,6651,1059,5288,65045,698,915,2025-11-18,2025-12-18 23:00:00,0.3581
5,5399,281739,268639,6616,5430,1054,5312,54559,651,896,2025-11-18,2025-12-18 23:00:00,0.3741
6,11279,187056,177961,4687,3686,722,4715,47538,585,649,2025-11-18,2025-12-18 23:00:00,0.3860
7,2825,163626,155949,3360,3692,625,4551,43997,413,538,2025-11-18,2025-12-18 23:00:00,0.3820
8,5232,144200,135506,2597,4486,1611,4972,25900,954,1004,2025-11-18,2025-12-18 23:00:00,1.1172
9,10894,138782,131221,2802,3894,865,4523,33482,592,717,2025-11-18,2025-12-18 23:00:00,0.6233



品类表已导出： ../data/processed/category_summary_v1.parquet


In [31]:
from pathlib import Path

CUTOFF_TIME = "2025-12-18 00:00:00"
LABEL_END = "2025-12-19 00:00:00"

print("观测窗口结束：", CUTOFF_TIME)
print("标签窗口结束：", LABEL_END)

con.execute(f"""
CREATE OR REPLACE TABLE user_item_summary AS

WITH observation_data AS (
    SELECT *
    FROM read_parquet('{parquet_sql_path}')
    WHERE time < TIMESTAMP '{CUTOFF_TIME}'
),

base_features AS (
    SELECT
        user_id,
        item_id,

        MIN(item_category)
            AS item_category,

        COUNT(*)
            AS total_actions,

        COUNT(*) FILTER (
            WHERE behavior_type = 1
        ) AS view_count,

        COUNT(*) FILTER (
            WHERE behavior_type = 2
        ) AS favorite_count,

        COUNT(*) FILTER (
            WHERE behavior_type = 3
        ) AS cart_count,

        COUNT(*) FILTER (
            WHERE behavior_type = 4
        ) AS purchase_count,

        MIN(time)
            AS first_interaction_time,

        MAX(time)
            AS last_interaction_time,

        COUNT(
            DISTINCT CAST(time AS DATE)
        ) AS interaction_days,

        COUNT(DISTINCT time)
            AS active_hours

    FROM observation_data

    GROUP BY
        user_id,
        item_id
),

purchase_labels AS (
    SELECT
        user_id,
        item_id,
        CAST(1 AS UTINYINT)
            AS purchase_next_horizon_label

    FROM read_parquet('{parquet_sql_path}')

    WHERE
        time >= TIMESTAMP '{CUTOFF_TIME}'
        AND time < TIMESTAMP '{LABEL_END}'
        AND behavior_type = 4

    GROUP BY
        user_id,
        item_id
)

SELECT
    base.user_id,
    base.item_id,
    base.item_category,
    base.total_actions,
    base.view_count,
    base.favorite_count,
    base.cart_count,
    base.purchase_count,
    base.first_interaction_time,
    base.last_interaction_time,
    base.interaction_days,
    base.active_hours,

    GREATEST(
        base.view_count - 1,
        0
    ) AS repeat_view_count,

    DATE_DIFF(
        'hour',
        base.last_interaction_time,
        TIMESTAMP '{CUTOFF_TIME}'
    ) AS recency_hours,

    COALESCE(
        labels.purchase_next_horizon_label,
        0
    ) AS purchase_next_horizon_label

FROM base_features AS base

LEFT JOIN purchase_labels AS labels
    ON base.user_id = labels.user_id
    AND base.item_id = labels.item_id
""")

观测窗口结束： 2025-12-18 00:00:00
标签窗口结束： 2025-12-19 00:00:00


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [32]:
# 用户—商品表基础校验
user_item_validation = con.execute(f"""
SELECT
    COUNT(*) AS user_item_rows,

    SUM(total_actions)
        AS restored_observation_actions,

    (
        SELECT COUNT(*)
        FROM read_parquet('{parquet_sql_path}')
        WHERE time < TIMESTAMP '{CUTOFF_TIME}'
    ) AS expected_observation_actions,

    SUM(purchase_next_horizon_label)
        AS positive_labels,

    ROUND(
        100.0
        * SUM(purchase_next_horizon_label)
        / COUNT(*),
        6
    ) AS positive_rate_pct,

    MIN(recency_hours)
        AS min_recency_hours,

    MAX(recency_hours)
        AS max_recency_hours

FROM user_item_summary
""").fetchdf()

# 检查预测日购买商品是否在候选集中
candidate_coverage = con.execute(f"""
WITH prediction_day_purchases AS (
    SELECT DISTINCT
        user_id,
        item_id

    FROM read_parquet('{parquet_sql_path}')

    WHERE
        time >= TIMESTAMP '{CUTOFF_TIME}'
        AND time < TIMESTAMP '{LABEL_END}'
        AND behavior_type = 4
)

SELECT
    COUNT(*) AS prediction_purchase_pairs,

    COUNT(summary.user_id)
        AS covered_purchase_pairs,

    ROUND(
        100.0
        * COUNT(summary.user_id)
        / COUNT(*),
        4
    ) AS candidate_coverage_pct

FROM prediction_day_purchases AS purchases

LEFT JOIN user_item_summary AS summary
    ON purchases.user_id = summary.user_id
    AND purchases.item_id = summary.item_id
""").fetchdf()

positive_examples = con.execute("""
SELECT *
FROM user_item_summary
WHERE purchase_next_horizon_label = 1
ORDER BY total_actions DESC
LIMIT 10
""").fetchdf()

print("===== 用户—商品表校验 =====")
display(user_item_validation)

print("\n===== 预测日购买候选覆盖率 =====")
display(candidate_coverage)

print("\n===== 正样本示例 =====")
display(positive_examples)

===== 用户—商品表校验 =====


,user_item_rows,restored_observation_actions,expected_observation_actions,positive_labels,positive_rate_pct,min_recency_hours,max_recency_hours
0,4546934,11881309.0,11881309,1028.0,0.022609,1,720



===== 预测日购买候选覆盖率 =====


,prediction_purchase_pairs,covered_purchase_pairs,candidate_coverage_pct
0,3151,1028,32.6246



===== 正样本示例 =====


,user_id,item_id,item_category,total_actions,view_count,favorite_count,cart_count,purchase_count,first_interaction_time,last_interaction_time,interaction_days,active_hours,repeat_view_count,recency_hours,purchase_next_horizon_label
0,71883073,58558840,1083,57,45,0,0,12,2025-11-19 12:00:00,2025-12-14 19:00:00,14,16,44,77,1
1,137998405,143865903,6307,57,53,0,3,1,2025-12-05 21:00:00,2025-12-17 09:00:00,8,12,52,15,1
2,131573117,8788846,10223,56,50,0,5,1,2025-11-24 10:00:00,2025-12-15 19:00:00,14,21,49,53,1
3,83570876,255708467,5027,56,53,2,1,0,2025-12-05 15:00:00,2025-12-17 15:00:00,13,26,52,9,1
4,71883073,380344970,1083,47,32,0,0,15,2025-11-19 12:00:00,2025-12-14 16:00:00,8,15,31,80,1
5,62073731,25237777,9396,44,42,1,1,0,2025-11-20 11:00:00,2025-12-17 23:00:00,13,17,41,1,1
6,70212546,249750060,1083,41,30,0,4,7,2025-11-25 10:00:00,2025-12-17 15:00:00,8,11,29,9,1
7,37780288,316255438,10961,35,34,0,1,0,2025-11-30 16:00:00,2025-12-17 19:00:00,7,8,33,5,1
8,108260948,71382075,7767,35,32,0,0,3,2025-11-23 16:00:00,2025-12-15 16:00:00,6,8,31,56,1
9,78462766,198971992,11135,34,31,0,0,3,2025-11-27 19:00:00,2025-12-16 17:00:00,10,11,30,31,1


In [33]:
candidate_diagnosis = con.execute(f"""
WITH prediction_purchases AS (
    SELECT DISTINCT
        user_id,
        item_id,
        item_category

    FROM read_parquet('{parquet_sql_path}')

    WHERE
        time >= TIMESTAMP '{CUTOFF_TIME}'
        AND time < TIMESTAMP '{LABEL_END}'
        AND behavior_type = 4
),

historical_pairs AS (
    SELECT DISTINCT
        user_id,
        item_id

    FROM read_parquet('{parquet_sql_path}')

    WHERE time < TIMESTAMP '{CUTOFF_TIME}'
),

historical_items AS (
    SELECT DISTINCT
        item_id

    FROM read_parquet('{parquet_sql_path}')

    WHERE time < TIMESTAMP '{CUTOFF_TIME}'
),

historical_user_categories AS (
    SELECT DISTINCT
        user_id,
        item_category

    FROM read_parquet('{parquet_sql_path}')

    WHERE time < TIMESTAMP '{CUTOFF_TIME}'
),

recent_pairs AS (
    SELECT DISTINCT
        user_id,
        item_id

    FROM read_parquet('{parquet_sql_path}')

    WHERE
        time >= TIMESTAMP '{CUTOFF_TIME}'
               - INTERVAL 7 DAY
        AND time < TIMESTAMP '{CUTOFF_TIME}'
),

recent_user_categories AS (
    SELECT DISTINCT
        user_id,
        item_category

    FROM read_parquet('{parquet_sql_path}')

    WHERE
        time >= TIMESTAMP '{CUTOFF_TIME}'
               - INTERVAL 7 DAY
        AND time < TIMESTAMP '{CUTOFF_TIME}'
),

diagnosis AS (
    SELECT
        purchases.*,

        CASE WHEN historical_pairs.user_id
                  IS NOT NULL
             THEN 1 ELSE 0
        END AS pair_seen_before,

        CASE WHEN historical_items.item_id
                  IS NOT NULL
             THEN 1 ELSE 0
        END AS item_seen_before,

        CASE WHEN historical_user_categories.user_id
                  IS NOT NULL
             THEN 1 ELSE 0
        END AS category_seen_by_user,

        CASE WHEN recent_pairs.user_id
                  IS NOT NULL
             THEN 1 ELSE 0
        END AS pair_seen_last_7d,

        CASE WHEN recent_user_categories.user_id
                  IS NOT NULL
             THEN 1 ELSE 0
        END AS category_seen_last_7d

    FROM prediction_purchases AS purchases

    LEFT JOIN historical_pairs
        ON purchases.user_id
           = historical_pairs.user_id
        AND purchases.item_id
           = historical_pairs.item_id

    LEFT JOIN historical_items
        ON purchases.item_id
           = historical_items.item_id

    LEFT JOIN historical_user_categories
        ON purchases.user_id
           = historical_user_categories.user_id
        AND purchases.item_category
           = historical_user_categories.item_category

    LEFT JOIN recent_pairs
        ON purchases.user_id
           = recent_pairs.user_id
        AND purchases.item_id
           = recent_pairs.item_id

    LEFT JOIN recent_user_categories
        ON purchases.user_id
           = recent_user_categories.user_id
        AND purchases.item_category
           = recent_user_categories.item_category
)

SELECT
    COUNT(*) AS total_purchase_pairs,

    SUM(pair_seen_before)
        AS pair_seen_before,

    ROUND(
        100.0 * SUM(pair_seen_before) / COUNT(*),
        4
    ) AS pair_seen_pct,

    SUM(item_seen_before)
        AS item_seen_before,

    ROUND(
        100.0 * SUM(item_seen_before) / COUNT(*),
        4
    ) AS item_seen_pct,

    SUM(category_seen_by_user)
        AS category_seen_by_user,

    ROUND(
        100.0
        * SUM(category_seen_by_user)
        / COUNT(*),
        4
    ) AS category_seen_pct,

    SUM(pair_seen_last_7d)
        AS pair_seen_last_7d,

    SUM(category_seen_last_7d)
        AS category_seen_last_7d

FROM diagnosis
""").fetchdf()

display(candidate_diagnosis)

,total_purchase_pairs,pair_seen_before,pair_seen_pct,item_seen_before,item_seen_pct,category_seen_by_user,category_seen_pct,pair_seen_last_7d,category_seen_last_7d
0,3151,1028.0,32.6246,2022.0,64.1701,2116.0,67.1533,861.0,1700.0


In [34]:
candidate_strategy_comparison = con.execute(f"""
WITH prediction_purchases AS (
    SELECT DISTINCT
        user_id,
        item_id,
        item_category

    FROM read_parquet('{parquet_sql_path}')

    WHERE
        time >= TIMESTAMP '{CUTOFF_TIME}'
        AND time < TIMESTAMP '{LABEL_END}'
        AND behavior_type = 4
),

observation_data AS (
    SELECT *
    FROM read_parquet('{parquet_sql_path}')
    WHERE time < TIMESTAMP '{CUTOFF_TIME}'
),

historical_pairs AS (
    SELECT DISTINCT
        user_id,
        item_id

    FROM observation_data
),

historical_user_categories AS (
    SELECT DISTINCT
        user_id,
        item_category

    FROM observation_data
),

recent_category_stats AS (
    SELECT
        user_id,
        item_category,
        COUNT(*) AS category_actions

    FROM observation_data

    WHERE
        time >= TIMESTAMP '{CUTOFF_TIME}'
               - INTERVAL 7 DAY

    GROUP BY
        user_id,
        item_category
),

recent_category_rank AS (
    SELECT
        user_id,
        item_category,

        ROW_NUMBER() OVER (
            PARTITION BY user_id
            ORDER BY
                category_actions DESC,
                item_category
        ) AS user_category_rank

    FROM recent_category_stats
),

item_stats AS (
    SELECT
        item_id,
        MIN(item_category) AS item_category,

        COUNT(*) AS total_actions,

        COUNT(*) FILTER (
            WHERE behavior_type = 3
        ) AS cart_count,

        COUNT(*) FILTER (
            WHERE behavior_type = 4
        ) AS purchase_count

    FROM observation_data

    GROUP BY item_id
),

item_rank AS (
    SELECT
        item_id,
        item_category,

        ROW_NUMBER() OVER (
            PARTITION BY item_category
            ORDER BY
                purchase_count DESC,
                cart_count DESC,
                total_actions DESC,
                item_id
        ) AS category_item_rank,

        ROW_NUMBER() OVER (
            ORDER BY
                purchase_count DESC,
                cart_count DESC,
                total_actions DESC,
                item_id
        ) AS global_item_rank

    FROM item_stats
),

purchase_flags AS (
    SELECT
        purchases.user_id,
        purchases.item_id,
        purchases.item_category,

        CASE
            WHEN historical_pairs.user_id
                 IS NOT NULL
            THEN 1 ELSE 0
        END AS historical_pair,

        CASE
            WHEN historical_user_categories.user_id
                 IS NOT NULL
            THEN 1 ELSE 0
        END AS historical_category,

        recent_category_rank.user_category_rank,

        item_rank.category_item_rank,
        item_rank.global_item_rank

    FROM prediction_purchases AS purchases

    LEFT JOIN historical_pairs
        ON purchases.user_id
           = historical_pairs.user_id
        AND purchases.item_id
           = historical_pairs.item_id

    LEFT JOIN historical_user_categories
        ON purchases.user_id
           = historical_user_categories.user_id
        AND purchases.item_category
           = historical_user_categories.item_category

    LEFT JOIN recent_category_rank
        ON purchases.user_id
           = recent_category_rank.user_id
        AND purchases.item_category
           = recent_category_rank.item_category

    LEFT JOIN item_rank
        ON purchases.item_id
           = item_rank.item_id
),

strategy_results AS (
    SELECT
        '仅历史用户—商品组合'
            AS strategy,

        COUNT(*) FILTER (
            WHERE historical_pair = 1
        ) AS covered_pairs

    FROM purchase_flags

    UNION ALL

    SELECT
        '历史组合＋近期Top5品类×品类Top20商品',

        COUNT(*) FILTER (
            WHERE
                historical_pair = 1
                OR (
                    user_category_rank <= 5
                    AND category_item_rank <= 20
                )
        )

    FROM purchase_flags

    UNION ALL

    SELECT
        '历史组合＋近期Top5品类×品类Top50商品',

        COUNT(*) FILTER (
            WHERE
                historical_pair = 1
                OR (
                    user_category_rank <= 5
                    AND category_item_rank <= 50
                )
        )

    FROM purchase_flags

    UNION ALL

    SELECT
        '历史组合＋近期Top10品类×品类Top50商品',

        COUNT(*) FILTER (
            WHERE
                historical_pair = 1
                OR (
                    user_category_rank <= 10
                    AND category_item_rank <= 50
                )
        )

    FROM purchase_flags

    UNION ALL

    SELECT
        '历史组合＋近期Top10品类×品类Top100商品',

        COUNT(*) FILTER (
            WHERE
                historical_pair = 1
                OR (
                    user_category_rank <= 10
                    AND category_item_rank <= 100
                )
        )

    FROM purchase_flags

    UNION ALL

    SELECT
        '历史组合＋近期Top10×Top100＋全局Top500',

        COUNT(*) FILTER (
            WHERE
                historical_pair = 1
                OR (
                    user_category_rank <= 10
                    AND category_item_rank <= 100
                )
                OR global_item_rank <= 500
        )

    FROM purchase_flags

    UNION ALL

    SELECT
        '理论上限：历史出现商品＋用户历史接触品类',

        COUNT(*) FILTER (
            WHERE
                historical_pair = 1
                OR (
                    historical_category = 1
                    AND category_item_rank IS NOT NULL
                )
        )

    FROM purchase_flags
)

SELECT
    strategy,
    covered_pairs,

    3151 - covered_pairs
        AS uncovered_pairs,

    ROUND(
        100.0 * covered_pairs / 3151,
        4
    ) AS coverage_pct

FROM strategy_results

ORDER BY coverage_pct
""").fetchdf()

display(candidate_strategy_comparison)

,strategy,covered_pairs,uncovered_pairs,coverage_pct
0,仅历史用户—商品组合,1028,2123,32.6246
1,历史组合＋近期Top5品类×品类Top20商品,1037,2114,32.9102
2,历史组合＋近期Top5品类×品类Top50商品,1045,2106,33.1641
3,历史组合＋近期Top10品类×品类Top50商品,1053,2098,33.4180
4,历史组合＋近期Top10品类×品类Top100商品,1069,2082,33.9257
5,历史组合＋近期Top10×Top100＋全局Top500,1096,2055,34.7826
6,理论上限：历史出现商品＋用户历史接触品类,1538,1613,48.8099


In [35]:
from pathlib import Path

# 只保留观测窗口特征，不包含预测标签
con.execute("""
CREATE OR REPLACE TABLE user_item_features AS
SELECT
    user_id,
    item_id,
    item_category,
    total_actions,
    view_count,
    favorite_count,
    cart_count,
    purchase_count,
    first_interaction_time,
    last_interaction_time,
    interaction_days,
    active_hours,
    repeat_view_count,
    recency_hours
FROM user_item_summary
""")

# 单独保存预测标签
con.execute("""
CREATE OR REPLACE TABLE user_item_labels AS
SELECT
    user_id,
    item_id,
    purchase_next_horizon_label
FROM user_item_summary
""")

# 校验特征表与标签表
step2_validation = con.execute("""
SELECT
    (SELECT COUNT(*)
     FROM user_item_features)
        AS feature_rows,

    (SELECT COUNT(*)
     FROM user_item_labels)
        AS label_rows,

    (SELECT SUM(total_actions)
     FROM user_item_features)
        AS restored_actions,

    (SELECT SUM(
        purchase_next_horizon_label
     )
     FROM user_item_labels)
        AS positive_labels
""").fetchdf()

display(step2_validation)

# 导出纯特征表
feature_output = Path(
    "../data/processed/user_item_features_v1.parquet"
)

if feature_output.exists():
    print("特征文件已经存在：", feature_output)
else:
    con.execute("""
    COPY user_item_features
    TO '../data/processed/user_item_features_v1.parquet'
    (FORMAT PARQUET, COMPRESSION ZSTD)
    """)

    print("特征表已导出：", feature_output)

# 标签单独保存，供步骤三使用
label_output = Path(
    "../data/processed/user_item_labels_v1.parquet"
)

if label_output.exists():
    print("标签文件已经存在：", label_output)
else:
    con.execute("""
    COPY user_item_labels
    TO '../data/processed/user_item_labels_v1.parquet'
    (FORMAT PARQUET, COMPRESSION ZSTD)
    """)

    print("标签表已导出：", label_output)

print(
    f"特征文件大小："
    f"{feature_output.stat().st_size / 1024**2:.2f} MB"
)

print(
    f"标签文件大小："
    f"{label_output.stat().st_size / 1024**2:.2f} MB"
)

,feature_rows,label_rows,restored_actions,positive_labels
0,4546934,4546934,11881309.0,1028.0


特征表已导出： ../data/processed/user_item_features_v1.parquet
标签表已导出： ../data/processed/user_item_labels_v1.parquet
特征文件大小：57.17 MB
标签文件大小：26.28 MB


In [36]:
from pathlib import Path
import pandas as pd

reports_dir = Path("../reports")
reports_dir.mkdir(parents=True, exist_ok=True)

# 获取各中间表实际行数
step2_table_summary = pd.DataFrame([
    {
        "中间表": "user_summary",
        "统计粒度": "每名用户一行",
        "主键": "user_id",
        "行数": con.execute(
            "SELECT COUNT(*) FROM user_summary"
        ).fetchone()[0],
        "状态": "已完成"
    },
    {
        "中间表": "item_summary",
        "统计粒度": "每件商品一行",
        "主键": "item_id",
        "行数": con.execute(
            "SELECT COUNT(*) FROM item_summary"
        ).fetchone()[0],
        "状态": "已完成"
    },
    {
        "中间表": "category_summary",
        "统计粒度": "每个品类一行",
        "主键": "item_category",
        "行数": con.execute(
            "SELECT COUNT(*) FROM category_summary"
        ).fetchone()[0],
        "状态": "已完成"
    },
    {
        "中间表": "time_hourly_summary",
        "统计粒度": "每小时一行",
        "主键": "time",
        "行数": con.execute(
            "SELECT COUNT(*) FROM time_summary"
        ).fetchone()[0],
        "状态": "已完成"
    },
    {
        "中间表": "time_daily_summary",
        "统计粒度": "每天一行",
        "主键": "action_date",
        "行数": con.execute(
            "SELECT COUNT(*) FROM daily_summary"
        ).fetchone()[0],
        "状态": "已完成"
    },
    {
        "中间表": "time_weekly_summary",
        "统计粒度": "每周一行",
        "主键": "week_start",
        "行数": con.execute(
            "SELECT COUNT(*) FROM weekly_summary"
        ).fetchone()[0],
        "状态": "已完成"
    },
    {
        "中间表": "user_item_features",
        "统计粒度": "每个历史用户—商品组合一行",
        "主键": "user_id + item_id",
        "行数": con.execute(
            "SELECT COUNT(*) FROM user_item_features"
        ).fetchone()[0],
        "状态": "已完成"
    },
    {
        "中间表": "user_item_labels",
        "统计粒度": "每个历史用户—商品组合一行",
        "主键": "user_id + item_id",
        "行数": con.execute(
            "SELECT COUNT(*) FROM user_item_labels"
        ).fetchone()[0],
        "状态": "已完成，步骤三使用"
    }
])

display(step2_table_summary)

# 保存校验汇总表
summary_csv_path = (
    reports_dir
    / "step2_intermediate_table_summary.csv"
)

step2_table_summary.to_csv(
    summary_csv_path,
    index=False,
    encoding="utf-8-sig"
)

# 生成正式步骤二报告
report_path = (
    reports_dir
    / "intermediate_table_design.md"
)

report_text = f"""# 多维度中间表构建与设计报告

## 1. 步骤目标

基于步骤一生成的标准化Parquet基础数据，
构建用户、商品、品类、时间和用户—商品维度中间表，
降低后续分析和特征计算的重复成本。

本步骤不训练模型。

## 2. 基础数据

| 指标 | 结果 |
|---|---:|
| 原始行为记录数 | 12,256,906 |
| 用户数 | 10,000 |
| 商品数 | 2,876,947 |
| 品类数 | 8,916 |
| 开始时间 | 2025-11-18 00:00:00 |
| 结束时间 | 2025-12-18 23:00:00 |

## 3. 中间表结构

| 中间表 | 统计粒度 | 主键 | 行数 | 状态 |
|---|---|---|---:|---|
| user_summary | 每名用户一行 | user_id | 10,000 | 已完成 |
| item_summary | 每件商品一行 | item_id | 2,876,947 | 已完成 |
| category_summary | 每个品类一行 | item_category | 8,916 | 已完成 |
| time_hourly_summary | 每小时一行 | time | 744 | 已完成 |
| time_daily_summary | 每天一行 | action_date | 31 | 已完成 |
| time_weekly_summary | 每周一行 | week_start | 5 | 已完成 |
| user_item_features | 每个历史用户—商品组合一行 | user_id + item_id | 4,546,934 | 已完成 |
| user_item_labels | 每个历史用户—商品组合一行 | user_id + item_id | 4,546,934 | 已完成，步骤三使用 |

## 4. 行为统计口径

行为编码定义：

- 1：浏览；
- 2：收藏；
- 3：加购；
- 4：购买。

重复行为没有直接删除，而是分别统计为：

- view_count；
- favorite_count；
- cart_count；
- purchase_count；
- repeat_view_count。

重复行为用于衡量用户兴趣强度和购买意向。

## 5. 各表主要指标

### 5.1 用户表

包括总交互数、四类行为次数、不同商品数、
不同品类数、活跃天数、首次和末次行为时间、
购买商品数及购买行为占比。

### 5.2 商品表

包括商品总流量、四类行为次数、独立交互用户数、
独立浏览用户数、独立收藏用户数、独立加购用户数、
独立购买用户数及购买行为占比。

### 5.3 品类表

包括品类总交互数、四类行为次数、商品数、
独立用户数、独立购买用户数和产生购买的商品数。

### 5.4 时间表

按照小时、日和周三个层级统计平台行为趋势，
包括总行为量、四类行为次数、活跃用户数和活跃商品数。

首周仅覆盖6天，末周仅覆盖4天，
因此周度比较不能只比较总量。

### 5.5 用户—商品表

观测窗口为2025年11月18日至12月17日，
以2025年12月18日作为预测窗口。

特征表与标签表已经物理分离：

- user_item_features只包含预测时点之前的行为特征；
- user_item_labels只保存预测窗口购买标签；
- 标签不得作为模型输入特征。

## 6. 一致性校验

| 校验项目 | 结果 |
|---|---:|
| 用户表还原行为数 | 12,256,906 |
| 商品表还原行为数 | 12,256,906 |
| 品类表还原行为数 | 12,256,906 |
| 日表还原行为数 | 12,256,906 |
| 周表还原行为数 | 12,256,906 |
| 观测窗口行为数 | 11,881,309 |
| 用户—商品特征表还原行为数 | 11,881,309 |
| 用户—商品正标签数 | 1,028 |
| 商品对应多个品类的异常数量 | 0 |

所有中间表均通过行数、行为总量、时间范围和主键粒度校验。

## 7. 输出文件

标准化和中间表文件保存在data/processed目录：

- user_behavior_standardized_v1.parquet
- user_summary_v1.parquet
- item_summary_v1.parquet
- category_summary_v1.parquet
- time_hourly_summary_v1.parquet
- time_daily_summary_v1.parquet
- time_weekly_summary_v1.parquet
- user_item_features_v1.parquet
- user_item_labels_v1.parquet

上述数据文件仅保存在本地，不上传GitHub。
GitHub仅保存代码、Notebook、设计文档和校验报告。

## 8. 步骤结论

步骤二已完成多维度中间表构建、字段设计、
数据一致性校验以及特征与标签分离。

下一步骤才进入探索性分析、滚动时间窗口特征
或模型训练数据准备。
"""

report_path.write_text(
    report_text,
    encoding="utf-8"
)

print("\n===== 步骤二报告生成结果 =====")
print("中间表汇总：", summary_csv_path)
print("设计与校验报告：", report_path)

,中间表,统计粒度,主键,行数,状态
0,user_summary,每名用户一行,user_id,10000,已完成
1,item_summary,每件商品一行,item_id,2876947,已完成
2,category_summary,每个品类一行,item_category,8916,已完成
3,time_hourly_summary,每小时一行,time,744,已完成
4,time_daily_summary,每天一行,action_date,31,已完成
5,time_weekly_summary,每周一行,week_start,5,已完成
6,user_item_features,每个历史用户—商品组合一行,user_id + item_id,4546934,已完成
7,user_item_labels,每个历史用户—商品组合一行,user_id + item_id,4546934,已完成，步骤三使用



===== 步骤二报告生成结果 =====
中间表汇总： ../reports/step2_intermediate_table_summary.csv
设计与校验报告： ../reports/intermediate_table_design.md
